# Day 12 — User Defined Functions (UDFs)

**What you will learn:**
- Row-at-a-time UDFs: `udf()` decorator, return type annotation, registration
- Why UDFs are a performance bottleneck (serialization, no Catalyst optimisation)
- Pandas (vectorized) UDFs: `pandas_udf`, Series→Series, Group Map
- UDFs in SQL: `spark.udf.register`
- When NOT to use a UDF

**Exam relevance:** DataFrame API (30%) — UDF type signatures, performance cost, and `pandas_udf` vs row UDF are tested.

In [ ]:
from pyspark.sql.functions import col, udf, pandas_udf
from pyspark.sql.types import StringType, IntegerType, DoubleType
import pandas as pd

## 1. Row-at-a-Time UDF

A standard UDF wraps a Python function and applies it row by row.

**Cost model:**
- Each row is serialized from JVM → Python, processed, then deserialized back
- Catalyst cannot inspect or optimise the function body
- 10-100× slower than equivalent built-in functions

> **Exam tip:** Always check whether a built-in function can replace your UDF. If yes, use the built-in.

In [ ]:
df = spark.createDataFrame([
    (1, "alice smith",   85, "engineering"),
    (2, "bob jones",     72, "marketing"),
    (3, "carol o'brien", 91, "engineering"),
    (4, "dave lee",      60, "sales"),
    (5, "eve chen",      78, "marketing"),
], ["id", "name", "score", "dept"])

# Define and register with the @udf decorator
@udf(returnType=StringType())
def title_case(s: str) -> str:
    return s.title() if s else None

df.withColumn("name_titled", title_case(col("name"))).show(truncate=False)

In [ ]:
# Alternative: udf() as a function — explicit return type required
def grade(score: int) -> str:
    if score is None: return None
    if score >= 90:   return "A"
    if score >= 75:   return "B"
    if score >= 60:   return "C"
    return "F"

grade_udf = udf(grade, StringType())

df.withColumn("grade", grade_udf(col("score"))).show()

## 2. UDFs in Spark SQL

Register a UDF so it can be called inside SQL strings.

In [ ]:
# Register for SQL use
spark.udf.register("grade_sql", grade, StringType())

df.createOrReplaceTempView("employees")
spark.sql("SELECT name, score, grade_sql(score) AS grade FROM employees").show()

## 3. Null Handling in UDFs

> **Exam tip:** UDFs receive `None` for null values. If your function doesn't guard against `None`, it will raise an exception at runtime.

In [ ]:
df_null = spark.createDataFrame([
    (1, 85),
    (2, None),
    (3, 72),
], ["id", "score"])

# Safe UDF — guards against None
@udf(returnType=StringType())
def safe_grade(score) -> str:
    if score is None:
        return "N/A"
    if score >= 90: return "A"
    if score >= 75: return "B"
    return "C"

df_null.withColumn("grade", safe_grade(col("score"))).show()

## 4. Pandas UDFs (Vectorized UDFs)

Pandas UDFs operate on columnar `pandas.Series` instead of one row at a time.

**Why faster:**
- Data transferred via Apache Arrow (columnar, zero-copy)
- Python function receives a `pd.Series`, applies vectorized operations
- 10-100× faster than row-at-a-time UDF for large datasets

| UDF type | Input | Output | Speed |
|---|---|---|---|
| Row UDF | Python scalar | Python scalar | Slowest |
| Pandas UDF (Series→Series) | `pd.Series` | `pd.Series` | Fast |
| Pandas UDF (Group Map) | `pd.DataFrame` | `pd.DataFrame` | Fast, arbitrary shape |

In [ ]:
# Series → Series pandas UDF
@pandas_udf(DoubleType())
def zscore(series: pd.Series) -> pd.Series:
    return (series - series.mean()) / series.std()

df.withColumn("score_z", zscore(col("score"))).show()

In [ ]:
# pandas_udf with string return type
@pandas_udf(StringType())
def upper_pd(s: pd.Series) -> pd.Series:
    return s.str.upper()

df.withColumn("name_upper", upper_pd(col("name"))).show(truncate=False)

## 5. Group Map Pandas UDF

Processes each group as a full `pd.DataFrame` — useful for complex per-group analytics.

In [ ]:
from pyspark.sql.types import StructType, StructField

schema = StructType([
    StructField("dept",  StringType(),  True),
    StructField("name",  StringType(),  True),
    StructField("score", IntegerType(), True),
    StructField("rank",  IntegerType(), True),
])

@pandas_udf(schema)
def rank_within_dept(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("score", ascending=False).reset_index(drop=True)
    pdf["rank"] = range(1, len(pdf) + 1)
    return pdf

# applyInPandas — explicit schema required
df.groupBy("dept").applyInPandas(rank_within_dept.func, schema=schema).show()

## 6. When NOT to Use a UDF

Built-in functions are always preferred because:
- Catalyst can push predicates through them
- No serialization overhead
- Null-safe by default

Replace UDFs with built-ins when possible:

| Instead of UDF for... | Use built-in |
|---|---|
| String case | `upper()`, `lower()`, `initcap()` |
| Conditional logic | `when().otherwise()` |
| Null coalescing | `coalesce()` |
| Math | `round()`, `abs()`, `sqrt()` |
| Date extraction | `year()`, `month()`, `dayofmonth()` |

In [ ]:
from pyspark.sql.functions import when

# Replace grade UDF with when().otherwise() — no serialization, Catalyst-optimised
df.withColumn("grade",
    when(col("score") >= 90, "A")
    .when(col("score") >= 75, "B")
    .when(col("score") >= 60, "C")
    .otherwise("F")
).show()

---

## Day 12 Checklist

- [ ] Defined a row UDF with `@udf(returnType=...)` decorator
- [ ] Registered a UDF for Spark SQL with `spark.udf.register`
- [ ] Handled `None` (null) inside a UDF safely
- [ ] Wrote a `@pandas_udf` (Series→Series) — faster than row UDF
- [ ] Know that row UDFs break Catalyst optimisation; pandas_udf uses Arrow
- [ ] Replaced a UDF with `when().otherwise()` — know when to avoid UDFs

**Next:** Day 13 — Window Functions (rank, lag, lead, running totals)